In [3]:
!pip install stable-baselines3[extra]
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import DDPG
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from stable_baselines3.common.noise import NormalActionNoise
from google.colab import drive
import pandas as pd
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [76]:
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory path
dir_path = "/content/drive/MyDrive/sp500_csv"

# Get the list of files in the directory
file_list = os.listdir(dir_path)

# Assuming you want to read the first CSV file in the directory:
file_path = os.path.join(dir_path, file_list[0])

# Load CSV into DataFrame
df = pd.read_csv(file_path)

# Convert 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')

# Define start and end dates (modify as needed)
start_date = "2022-06-30"  # YYYY-MM-DD format
end_date = "2022-12-12"    # YYYY-MM-DD format

# Filter data for the given date range
df_filtered = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# Display the first few rows of the filtered DataFrame
df_filtered.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Date,Low,Open,Volume,High,Close,Adjusted Close
6858,2022-06-30,140.809998,142.289993,1299400,143.479996,141.479996,140.608490
6859,2022-07-01,139.300003,141.850006,985400,143.470001,143.139999,142.258270
6860,2022-07-05,138.240005,142.250000,1030100,142.580002,140.710007,139.843246
6861,2022-07-06,139.539993,140.289993,1525100,141.729996,139.839996,138.978592
6862,2022-07-07,140.070007,140.449997,942100,141.919998,140.789993,139.922729


In [75]:
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory path
dir_path = "/content/drive/MyDrive/sp500_csv"

# Get the list of files in the directory
file_list = os.listdir(dir_path)

# Assuming you want to read the first CSV file in the directory:
file_path = os.path.join(dir_path, file_list[0])

# Load CSV into DataFrame
df = pd.read_csv(file_path)
df_filtered.head()

# Convert 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')

# Define start and end dates (modify as needed)
start_date = "2022-06-30"  # YYYY-MM-DD format
end_date = "2022-12-12"    # YYYY-MM-DD format

# Filter data for the given date range
df_filtered = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# Check for missing values
print("Missing values in the dataset:")
print(df_filtered.isnull().sum())

# Drop rows with missing values
df_filtered = df_filtered.dropna()

# Add additional features (e.g., Moving Averages, Returns)
df_filtered['SMA_10'] = df_filtered['Close'].rolling(window=10).mean()
df_filtered['SMA_50'] = df_filtered['Close'].rolling(window=50).mean()
df_filtered['Daily_Return'] = df_filtered['Close'].pct_change()

# Drop rows with NaN values created by rolling calculations
df_filtered = df_filtered.dropna()

# Reset the index of df_filtered
df_filtered = df_filtered.reset_index(drop=True)
# Drop the 'Date' column
df_filtered = df_filtered.drop(columns=['Date'])

print("\nPreprocessed DataFrame:")
print(df_filtered.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Missing values in the dataset:
Date              0
Low               0
Open              0
Volume            0
High              0
Close             0
Adjusted Close    0
dtype: int64

Preprocessed DataFrame:
Empty DataFrame
Columns: [Low, Open, Volume, High, Close, Adjusted Close, SMA_10, SMA_50, Daily_Return]
Index: []


In [20]:
!pip install shimmy

Model Building

In [21]:
import gym
import numpy as np
import pandas as pd
from gym import spaces
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

# Create a Custom Trading Environment for DDPG
class CustomTradingEnv(gym.Env):
    def __init__(self, df):
        super(CustomTradingEnv, self).__init__()
        self.df = df
        self.current_step = 0
        self.initial_balance = 10000  # Starting cash
        self.cash = self.initial_balance
        self.shares_held = 0
        self.max_steps = len(df) - 1
        # Action space: Continuous action between -1 and 1
        # -1: Sell all shares
        # 0: Hold
        # 1: Buy with all available cash
        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)
        # Observation space: [Current Price, Moving Average, RSI, Cash Available, Shares Held]
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(5,), dtype=np.float32)

    def reset(self):
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares_held = 0
        return self._next_observation()

    def _next_observation(self):
        obs = np.array([
            self.df.iloc[self.current_step]["Close"],
            self.df.iloc[self.current_step]["SMA_50"],
            self.df.iloc[self.current_step]["RSI"],
            self.cash,
            self.shares_held
        ], dtype=np.float32)
        return obs

    def step(self, action):
        current_price = self.df.iloc[self.current_step]["Close"]
        action = action[0]  # Extract the scalar value from the action array

        # Buy/Sell based on the continuous action
        if action > 0:  # Buy
            max_shares_to_buy = self.cash // current_price
            shares_to_buy = int(action * max_shares_to_buy)
            self.shares_held += shares_to_buy
            self.cash -= shares_to_buy * current_price
        elif action < 0:  # Sell
            shares_to_sell = int(-action * self.shares_held)
            self.shares_held -= shares_to_sell
            self.cash += shares_to_sell * current_price

        self.current_step += 1
        done = self.current_step >= self.max_steps
        reward = self.cash + (self.shares_held * current_price) - self.initial_balance

        return self._next_observation(), reward, done, {}

# Example usage
data = {
    'Close': np.random.rand(100) * 100,
    'SMA_50': np.random.rand(100) * 100,
    'RSI': np.random.rand(100) * 100
}
df = pd.DataFrame(data)
env = CustomTradingEnv(df)

# Define action noise for exploration
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Initialize the DDPG model with adjusted hyperparameters
model = DDPG(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,  # Reduced learning rate
    buffer_size=100000,
    batch_size=128,
    gamma=0.99,
    tau=0.005,
    action_noise=action_noise,
    tensorboard_log="./ddpg_trading_tensorboard/"
)

# Train the model
model.learn(total_timesteps=10000, log_interval=10)

# Evaluate the model and calculate total rewards
obs = env.reset()
done = False
total_rewards = 0
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    total_rewards += reward

print("Total Rewards:", total_rewards)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Logging to ./ddpg_trading_tensorboard/DDPG_1
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 99        |
|    ep_rew_mean     | 5.55e+05  |
| time/              |           |
|    episodes        | 10        |
|    fps             | 41        |
|    time_elapsed    | 23        |
|    total_timesteps | 990       |
| train/             |           |
|    actor_loss      | -3.96e+04 |
|    critic_loss     | 1.87e+08  |
|    learning_rate   | 0.0001    |
|    n_updates       | 889       |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 99        |
|    ep_rew_mean     | 2.77e+05  |
| time/              |           |
|    episodes        | 20        |
|    fps             | 44        |
|    time_elapsed    | 44        |
|    total_timesteps | 1980      |
| train/             |           |
|    actor_loss      | -5.67e+04 |
|    critic_loss     | 1.24e+09  |
|    learn

Model Training- Adjusting Hyperparameters

In [80]:
# Define action noise for exploration
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.2 * np.ones(n_actions))

# Adjusted DDPG hyperparameters for better performance
model = DDPG(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-5,  # Lower learning rate for stable training
    buffer_size=500000,  # Increased buffer size
    batch_size=256,  # Larger batch size for better learning
    gamma=0.995,  # Higher discount factor for long-term rewards
    tau=0.002,  # Slower target update rate
    action_noise=action_noise,  # More exploration
    policy_kwargs=dict(net_arch=[400, 300]),  # Larger network for learning
    tensorboard_log="./ddpg_trading_tensorboard/"
)

# Train the model
print("Starting Training...")
model.learn(total_timesteps=50000, log_interval=10)
print("Training Completed!")

# Save the trained model
model.save("ddpg_trading_model")

# Evaluate the model and calculate total rewards
obs = env.reset()
done = False
total_rewards = 0
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    total_rewards += reward

print("Total Rewards after Training:", total_rewards)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Starting Training...
Logging to ./ddpg_trading_tensorboard/DDPG_15


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 99        |
|    ep_rew_mean     | 8.41e+05  |
| time/              |           |
|    episodes        | 10        |
|    fps             | 38        |
|    time_elapsed    | 25        |
|    total_timesteps | 990       |
| train/             |           |
|    actor_loss      | -2.98e+04 |
|    critic_loss     | 2e+09     |
|    learning_rate   | 3e-05     |
|    n_updates       | 889       |
----------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 99       |
|    ep_rew_mean     | 4.2e+05  |
| time/              |          |
|    episodes        | 20       |
|    fps             | 35       |
|    time_elapsed    | 55       |
|    total_timesteps | 1980     |
| train/             |          |
|    actor_loss      | -5.3e+04 |
|    critic_loss     | 1.01e+10 |
|    learning_rate   | 3e-05    |
|    n_updates       | 1879     |

In [85]:
risk_free_rate = 0.02 / 252  # Daily risk-free rate (252 trading days per year)
df_filtered['RiskFreeRate'] = risk_free_rate

# Calculate returns
df_filtered['Return'] = df_filtered['Close'].pct_change()

# Drop any rows with NaN values that might have been created in the process
df_filtered = df_filtered.dropna()

# Calculate excess returns
df_filtered['Excess Return'] = df_filtered['Return'] - df_filtered['RiskFreeRate']

# Calculate the mean and standard deviation of excess returns
mean_excess_return = df_filtered['Excess Return'].mean()
std_excess_return = df_filtered['Excess Return'].std()

# Calculate the Sharpe ratio (assuming annualized)
sharpe_ratio = (mean_excess_return / std_excess_return) * np.sqrt(252)  # 252 trading days in a year

print(f"Sharpe Ratio: {sharpe_ratio:.4f}")

Sharpe Ratio: 1.5411


Sharpe Ratio Calculation

In [92]:
risk_free_rate = 0.02 / 252  # Daily risk-free rate (252 trading days per year)
df_filtered['RiskFreeRate'] = risk_free_rate

# Calculate returns
df_filtered['Return'] = df_filtered['Close'].pct_change()

# Drop any rows with NaN values that might have been created in the process
df_filtered = df_filtered.dropna()

# Calculate excess returns
df_filtered['Excess Return'] = df_filtered['Return'] - df_filtered['RiskFreeRate']

# Calculate the mean and standard deviation of excess returns
mean_excess_return = df_filtered['Excess Return'].mean()
std_excess_return = df_filtered['Excess Return'].std()

# Calculate the Sharpe ratio (assuming annualized)
sharpe_ratio = (mean_excess_return / std_excess_return) * np.sqrt(252)  # 252 trading days in a year

print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
# %%
# Declare and define data as a copy of df_filtered
data = df_filtered.copy()

# Ensure the 'Date' column is in datetime format
data['Date'] = pd.to_datetime(data['Date'])

# Calculate cumulative returns for portfolio performance
data['Cumulative Return'] = (1 + data['Return']).cumprod()

# Calculate drawdowns
data['Peak'] = data['Cumulative Return'].cummax()  # Rolling maximum of cumulative returns
data['Drawdown'] = (data['Cumulative Return'] - data['Peak']) / data['Peak']  # Drawdown formula

# Calculate maximum drawdown
max_drawdown = data['Drawdown'].min()
max_drawdown_date = data.loc[data['Drawdown'].idxmin(), 'Date']

# Print key metrics
final_cumulative_return = data['Cumulative Return'].iloc[-1]
print(f"Final Cumulative Return: {final_cumulative_return:.4f}")
print(f"Maximum Drawdown: {max_drawdown:.4f} (on {max_drawdown_date})")


Sharpe Ratio: 1.6904
Final Cumulative Return: 1.1875
Maximum Drawdown: -0.1135 (on 2022-09-30 00:00:00)
